# EPC Data Quality Audit: Camden, London
## 1. Data profiling

Source: Domestic Energy Performance Certificates, Camden, Jan 2012 - Jun 2026,
downloaded from get-energy-performance-data.communities.gov.uk (data not included
in repo due to licence).

In [2]:
import pandas as pd

df = pd.read_csv("../data/raw/camden_certificates.csv", low_memory=False)
print(df.shape)

(102185, 93)


In [3]:
df.head()

,certificate_number,address1,address2,address3,postcode,posttown,address,constituency,constituency_label,local_authority,...,walls_env_eff,wind_turbine_count,windows_description,windows_energy_eff,windows_env_eff,floor_env_eff,region,country,uprn,uprn_source
0,9637-2800-7390-9224-9051,"3, Agar Grove",NaN,NaN,NW1 9SL,LONDON,"3, Agar Grove",E14001290,Holborn and St Pancras,E09000007,...,Very Poor,0.0,Single glazed,Very Poor,Very Poor,NaN,E12000007,England,5055943.0,Energy Assessor
1,0060-2855-6531-9721-2181,"18, Brassey Road",NaN,NaN,NW6 2BE,LONDON,"18, Brassey Road",E14001265,Hampstead and Highgate,E09000007,...,Poor,0.0,Single glazed,Very Poor,Very Poor,NaN,E12000007,England,5003408.0,Energy Assessor
2,0125-3852-7281-9794-8445,7 Pickstock Court,155 Gray's Inn Road,NaN,WC1X 8UE,LONDON,"7 Pickstock Court, 155 Gray's Inn Road",E14001290,Holborn and St Pancras,E09000007,...,Very Good,0.0,High performance glazing,Very Good,Very Good,NaN,E12000007,England,5172370.0,Energy Assessor
3,0248-4906-7239-2994-6924,14b Agamemnon Road,NaN,NaN,NW6 1DY,LONDON,14b Agamemnon Road,E14001265,Hampstead and Highgate,E09000007,...,Very Poor,0.0,Single glazed,Very Poor,Very Poor,NaN,E12000007,England,NaN,NaN
4,0296-2873-6747-9592-3255,Rear Garden Flat,"61, Belsize Park Gardens",NaN,NW3 4JN,LONDON,"Rear Garden Flat, 61, Belsize Park Gardens",E14001265,Hampstead and Highgate,E09000007,...,Very Poor,0.0,Partial double glazing,Poor,Poor,NaN,E12000007,England,5115246.0,Energy Assessor


In [10]:
df.columns.tolist()

['certificate_number',
 'address1',
 'address2',
 'address3',
 'postcode',
 'posttown',
 'address',
 'constituency',
 'constituency_label',
 'local_authority',
 'local_authority_label',
 'built_form',
 'co2_emiss_curr_per_floor_area',
 'co2_emissions_current',
 'co2_emissions_potential',
 'construction_age_band',
 'current_energy_efficiency',
 'current_energy_rating',
 'energy_consumption_current',
 'energy_consumption_potential',
 'energy_tariff',
 'environment_impact_current',
 'environment_impact_potential',
 'extension_count',
 'fixed_lighting_outlets_count',
 'flat_storey_count',
 'flat_top_storey',
 'floor_description',
 'floor_energy_eff',
 'floor_height',
 'floor_level',
 'glazed_area',
 'glazed_type',
 'heat_loss_corridor',
 'heating_cost_current',
 'heating_cost_potential',
 'hot_water_cost_current',
 'hot_water_cost_potential',
 'hot_water_energy_eff',
 'hot_water_env_eff',
 'hotwater_description',
 'inspection_date',
 'lighting_cost_current',
 'lighting_cost_potential',
 'l

## 2. Completeness: missing values

In [16]:
missing = df.isna().sum().sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(1)
missing_pct.head(20)

sheating_env_eff            100.0
sheating_energy_eff         100.0
floor_env_eff                96.4
floor_energy_eff             96.4
secondheat_description       91.7
address3                     82.8
roof_env_eff                 63.2
roof_energy_eff              63.2
unheated_corridor_length     57.0
heat_loss_corridor           35.1
glazed_area                  34.9
glazed_type                  34.9
photo_supply                 28.5
mains_gas_flag               28.4
address2                     23.6
mechanical_ventilation       11.6
number_heated_rooms          11.6
solar_water_heating_flag     11.6
number_habitable_rooms       11.6
extension_count              11.6
dtype: float64

In [17]:
critical = ["uprn", "certificate_number", "postcode", "lodgement_date",
            "total_floor_area", "current_energy_rating", "construction_age_band", "tenure"]
missing_pct[critical].sort_values(ascending=False)

uprn                     9.1
tenure                   4.7
postcode                 0.0
certificate_number       0.0
lodgement_date           0.0
total_floor_area         0.0
current_energy_rating    0.0
construction_age_band    0.0
dtype: float64

### Findings: completeness

The core certificate fields are in good health: postcode, certificate_number,
lodgement_date, total_floor_area and current_energy_rating are all effectively
~100% complete across 102,185 records.

The gaps are concentrated where they hurt most:

- **9.1% of certificates (~9,300) have no UPRN**, the unique property identifier.
  These records cannot be reliably linked to a specific home, which undermines
  deduplication, property history tracking, and any address-level analysis.
- **11.6% of records (~11,900 homes) are missing basic property facts** such as
  number_habitable_rooms and number_heated_rooms. Five fields share this exact
  rate, suggesting a structural cause (e.g. certain assessment types omitting
  them) rather than random entry errors: to be investigated.
- **4.7% lack tenure** (owner-occupied vs rented), a gap with regulatory weight
  since minimum energy standards for rentals depend on knowing tenure.
- Two columns (sheating_env_eff, sheating_energy_eff) are 100% empty: dead
  columns carried in the extract.
- High missingness in address2 (23.6%) and address3 (82.8%) **is expected**, not a
  defect: most properties simply need fewer address lines.